# MeChess 10/10: the human games for the classifier test (CPU notebook)

An adapter around `chessme humanlike-humangames`. For starting positions of notebook 6's test set TB (early in the game, time left on both clocks, one per game, from games the adapters never saw) it fetches the **real continuation** of each game from the public Lichess games export by game id: moves and clocks, written in the same format as the generated games of notebook 9. These are the human side of the comparison with generated games (the classifier two-sample test, and the stimuli of the forced-choice study). Only derived numbers are kept: no account names.

**Before you run:** add notebook 6's output (the test sets) as an input (*Add Input -> Notebook output files*); *Accelerator -> None*, *Internet -> On*; **Save Version -> Save & Run All (Commit)**. Requests go to Lichess one at a time with a pause (300 ids each), so 20,000 games take about ten minutes; the notebook waits and retries when Lichess asks it to slow down.

In [ ]:
import glob, os, shutil, subprocess, sys

REPO_URL = "https://github.com/<you>/MeChess.git"      # <- put your repository URL here (not needed when the repository is attached as a dataset)
GAMES = 20000                                           # starting positions (one per game); fewer when the test set has fewer
SPLIT = "test"                                          # games of this part of the adapter split (the adapters were trained on the others)
OUT = "/kaggle/working/humanlike" if os.path.exists("/kaggle") else "humanlike_local"
INPUT_DIR = ""                                          # notebook 6's output folder; empty: search /kaggle/input

def sh(*args):
    """Run a command and stream its output; stop the notebook if it fails."""
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"failed: {' '.join(args)}")

if not os.path.exists("chessme"):
    attached = sorted(glob.glob("/kaggle/input/**/chessme/__init__.py", recursive=True), key=lambda f: "/datasets/" not in f)   # prefer the attached repository dataset
    if attached:
        shutil.copytree(os.path.dirname(os.path.dirname(attached[0])), "MeChess")
    else:                                                           # Kaggle: fetch the repository (Internet must be on)
        if "<you>" in REPO_URL:
            raise SystemExit("Attach the repository as a dataset, or set REPO_URL.")
        sh("git", "clone", "--depth", "1", REPO_URL, "MeChess")
    os.chdir("MeChess")
sh(sys.executable, "-m", "pip", "-q", "install", "chess==1.11.2", "numpy", "pyyaml", "requests", "zstandard")
CLI = [sys.executable, "-m", "chessme"]              # every step below is one `chessme` command: the same ones you run on a laptop

In [ ]:
root = INPUT_DIR or "/kaggle/input"
TB = next(iter(sorted(glob.glob(root + "/**/testset_TB.jsonl.gz", recursive=True))), None)
print("test set:", TB)
if not TB:
    raise SystemExit("Add notebook 6's output (testset_TB.jsonl.gz) as an input.")
os.makedirs(OUT, exist_ok=True)

## Step 1. The real continuations

In [ ]:
sh(*CLI, "humanlike-humangames", "--testset", TB, "--split", SPLIT, "--games", str(GAMES), "--out", f"{OUT}/games_human.jsonl")

## Step 2. Statistics of the human games (the generated games of notebook 9 are summarised the same way)

In [ ]:
sh(*CLI, "humanlike-gamestats", "--games", f"human={OUT}/games_human.jsonl", "--out", f"{OUT}/gamestats_human.md")